# 07-02a — Prepare Class-Weight Inputs (for 07-02)

Run this **once before 07-02** (on Colab or locally). It rebuilds the inputs
that 07-02 expects in ``./results_dedicated_lr_mlp`` and never trains an MLP,
so it is cheap compared with re-running all of 07-01:

1. **Data caches** — ``data_<drug>.pkl`` for all 10 drugs (70/15/15
   species-stratified split + log1p/standardize, identical to 07-01).
   Already-present files are skipped, so the notebook is resumable.
2. **LR results** — ``lr_results.pkl`` with 07-01's exact
   ``train_eval_lr`` protocol (15-value C grid, 3-fold CV threshold tuning),
   checkpointed per drug under ``lr_ckpt/``.
3. **MLP results** — if ``mlp_results.pkl`` is missing it is reconstructed
   from the exported ``07-01-results_dedicated_lr_mlp/results_balacc.csv``
   and ``results_auc.csv`` (Test metrics only; validation metrics would
   require a full 07-01 re-run).

At the end the notebook verifies that all inputs are present.


In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
# =============================================================================
# 0. SETUP
# =============================================================================
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted")
    IN_COLAB = True
except ImportError:
    print("Running locally")
    IN_COLAB = False

if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    CANDIDATES = [
        Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet"),
        Path("/mnt/sda-home/asd/Projects/Flower/DataSet/Dryad-DataSet"),
        Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet"),
    ]
    DRYAD = next((p for p in CANDIDATES if (p / "Processed").exists()), CANDIDATES[0])
assert (DRYAD / "Processed").exists(), f"Processed data not found under {DRYAD}"

SAVE_DIR = Path("./results_dedicated_lr_mlp")
SAVE_DIR.mkdir(exist_ok=True)
CSV_DIR = Path("./07-01-results_dedicated_lr_mlp")
assert CSV_DIR.exists(), f"Missing {CSV_DIR} (07-01 exports)"

DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]
C_GRID = np.linspace(5e-5, 1e-3, 15)
THRESHOLDS = np.linspace(0.05, 0.95, 91)


def safe_name(drug):
    return drug.replace(" ", "_").replace("-", "_")


print(f"DRYAD:    {DRYAD}")
print(f"SAVE_DIR: {SAVE_DIR.resolve()}")
print(f"CSV_DIR:  {CSV_DIR.resolve()}")


In [ ]:
# =============================================================================
# 1. LOADER (identical to 07-01)
# =============================================================================
def load_drug_data(drug_name):
    """Load a single drug's data from all 4 sites, concatenated."""
    frames = []
    dir_name = drug_name.replace(" ", "_")
    for site in SITES:
        p = DRYAD / "Processed" / f"Proc_{site}" / dir_name / "data.csv"
        if not p.exists(): continue
        df = pd.read_csv(p)
        df["site"] = site
        frames.append(df)
    df_all = pd.concat(frames, ignore_index=True)
    bin_cols = [c for c in df_all.columns if c.startswith("bin_")]
    X = df_all[bin_cols].to_numpy(dtype="float32")
    y = df_all["label"].to_numpy(dtype="int64")
    species = df_all["species"].to_numpy()
    return X, y, species


In [ ]:
# =============================================================================
# 2. DATA CACHES — 70/15/15 split + preprocessing (same as 07-01)
# =============================================================================
for drug in DRUGS_10:
    out_path = SAVE_DIR / f"data_{safe_name(drug)}.pkl"
    if out_path.exists():
        print(f"  {drug:35s}  cache exists, skipping")
        continue

    X, y, species = load_drug_data(drug)
    idx = np.arange(len(y)).reshape(-1, 1)
    idx_trval, idx_test, _, _ = stratified_species_drug_split(
        idx, y, species=species, test_size=0.15, random_state=SEED)
    idx_trval = idx_trval.flatten().astype(int)
    idx_test = idx_test.flatten().astype(int)

    val_frac = 0.15 / 0.85
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx_trval.reshape(-1, 1), y[idx_trval], species=species[idx_trval],
        test_size=val_frac, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)

    X_train_raw, y_train = X[idx_train], y[idx_train]
    X_val_raw,   y_val   = X[idx_val], y[idx_val]
    X_test_raw,  y_test  = X[idx_test], y[idx_test]

    state = fit_input_transform(X_train_raw, "log1p+standardize")
    saved = {
        "X_train_pp": apply_input_transform(X_train_raw, state), "y_train": y_train,
        "X_val_pp":   apply_input_transform(X_val_raw, state),   "y_val": y_val,
        "X_test_pp":  apply_input_transform(X_test_raw, state),  "y_test": y_test,
        "state": state, "idx_train": idx_train, "idx_val": idx_val, "idx_test": idx_test,
    }
    with open(out_path, "wb") as f:
        pickle.dump(saved, f)
    print(f"  {drug:35s}  saved  train={len(y_train)}  val={len(y_val)}  test={len(y_test)}"
          f"  R_train={int(y_train.sum())}")


In [ ]:
# =============================================================================
# 3. LR RESULTS — 07-01's exact protocol
# =============================================================================
def train_eval_lr(X_train, y_train, test_sets):
    """GridSearchCV(C) + CV threshold tuning. Returns {test_name: {BalAcc, AUC}}."""
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_c = grid.best_params_["C"]

    cv_proba = cross_val_predict(
        LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]

    lr = LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                            class_weight="balanced", max_iter=5000, random_state=SEED)
    lr.fit(X_train, y_train)

    results = {}
    for name, (X_te, y_te) in test_sets.items():
        proba = lr.predict_proba(X_te)[:, 1]
        preds = proba >= best_t
        results[name] = {
            "BalAcc": balanced_accuracy_score(y_te, preds),
            "AUC": roc_auc_score(y_te, proba),
            "Threshold": best_t, "Best_Param": f"C={best_c:.2e}",
        }
    return results


In [ ]:
# =============================================================================
# 4. RUN LR (checkpointed per drug)
# =============================================================================
LR_CKPT = SAVE_DIR / "lr_ckpt"
LR_CKPT.mkdir(exist_ok=True)

if (SAVE_DIR / "lr_results.pkl").exists():
    print("lr_results.pkl exists - skipping LR recompute")
else:
    for drug in DRUGS_10:
        ckpt = LR_CKPT / f"lr_{safe_name(drug)}.pkl"
        if ckpt.exists():
            print(f"  {drug:35s}  LR checkpoint exists, skipping")
            continue
        with open(SAVE_DIR / f"data_{safe_name(drug)}.pkl", "rb") as f:
            d = pickle.load(f)
        test_sets = {"Val": (d["X_val_pp"], d["y_val"]),
                     "Test": (d["X_test_pp"], d["y_test"])}
        res = train_eval_lr(d["X_train_pp"], d["y_train"], test_sets)
        with open(ckpt, "wb") as f:
            pickle.dump(res, f)
        print(f"  {drug:35s}  Test BalAcc={res['Test']['BalAcc']:.4f}  "
              f"AUC={res['Test']['AUC']:.4f}  threshold={res['Test']['Threshold']:.2f}")

    lr_results = {}
    for drug in DRUGS_10:
        with open(LR_CKPT / f"lr_{safe_name(drug)}.pkl", "rb") as f:
            lr_results[drug] = pickle.load(f)
    assert len(lr_results) == len(DRUGS_10)
    with open(SAVE_DIR / "lr_results.pkl", "wb") as f:
        pickle.dump(lr_results, f)
    print(f"\nSaved lr_results.pkl for {len(lr_results)} drugs")


In [ ]:
# =============================================================================
# 5. MLP RESULTS — reconstruct Test metrics from 07-01's exported CSVs
# =============================================================================
mlp_path = SAVE_DIR / "mlp_results.pkl"
if mlp_path.exists():
    print("mlp_results.pkl exists - skipping reconstruction")
else:
    ba = pd.read_csv(CSV_DIR / "results_balacc.csv")
    au = pd.read_csv(CSV_DIR / "results_auc.csv")
    mlp = {}
    for _, r in ba.iterrows():
        drug = r["Drug"]
        auc_val = au.loc[au["Drug"] == drug, "MLP"].iloc[0]
        mlp[drug] = {"Test": {"BalAcc": float(r["MLP"]), "AUC": float(auc_val)}}
    assert len(mlp) == len(DRUGS_10), len(mlp)
    with open(mlp_path, "wb") as f:
        pickle.dump(mlp, f)
    print("Reconstructed mlp_results.pkl from CSV exports (Test metrics only).")
    print("NOTE: validation metrics and per-drug params require a full 07-01 run.")


In [ ]:
# =============================================================================
# 6. VERIFY ALL INPUTS
# =============================================================================
missing = [d for d in DRUGS_10 if not (SAVE_DIR / f"data_{safe_name(d)}.pkl").exists()]
assert not missing, f"Missing data caches: {missing}"
for name in ["lr_results.pkl", "mlp_results.pkl"]:
    assert (SAVE_DIR / name).exists(), f"Missing {name}"

with open(SAVE_DIR / "lr_results.pkl", "rb") as f:
    lr_results = pickle.load(f)
with open(SAVE_DIR / "mlp_results.pkl", "rb") as f:
    mlp_results = pickle.load(f)

print(f"data caches : {len(DRUGS_10)}/10")
print(f"LR results  : {len(lr_results)}/10")
print(f"MLP results : {len(mlp_results)}/10")
print("\nReady for 07-02-Dedicated-MLP-ClassWeight.ipynb")
